# LSTM Stock Prediction Model

This notebook demonstrates how to load the normalized data and train an LSTM neural network to predict short-term stock trends using PyTorch.


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## 1. Load Data & Create Target
We load the pre-calculated Z-scores. We want to predict if the price will be **higher 15 minutes from now**.


In [ ]:
# Load the normalized data for a single stock (Proof of Concept)
print("Loading data...")
df = pd.read_parquet('normalized_data/RELIANCE_minute.parquet')

# To speed up this initial test, we will only use the last 50,000 minutes (approx 6 months)
df = df.iloc[-50000:].copy()

# Define the Target Variable (1 if price goes UP in 15 mins, 0 otherwise)
future_periods = 15
df['Target'] = (df['close'].shift(-future_periods) > df['close']).astype(int)

# Drop the last 15 rows which won't have a target
df.dropna(inplace=True)

print(f"Data shape: {df.shape}")
print(f"Target distribution (1=Up, 0=Down): \n{df['Target'].value_counts(normalize=True)}")


## 2. Sequence Generation
LSTMs require input data in sequences (lookbacks). We will look back 60 minutes for each prediction.


In [ ]:
sequence_length = 60 # Look back 60 minutes

features = ['open', 'high', 'low', 'close', 'volume']
data_values = df[features].values
target_values = df['Target'].values

X, y = [], []
print("Generating sequences... this might take a minute.")
for i in range(len(df) - sequence_length):
    X.append(data_values[i : i + sequence_length])
    y.append(target_values[i + sequence_length - 1]) # Target aligned with the end of the sequence

X = np.array(X)
y = np.array(y)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Split chronologically (Do NOT shuffle time-series data)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]


## 3. PyTorch Dataset and Model Architecture


In [ ]:
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 128
train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(StockDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

# Define the LSTM Model
class TradingLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(TradingLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # batch_first=True expects input shape (batch, seq, feature)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        
        # Output layer
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        
        out, _ = self.lstm(x, (h0, c0))
        
        # Decode the hidden state of the last time step
        out = out[:, -1, :] 
        out = self.fc(out)
        return out.squeeze()

# Instantiate the model
model = TradingLSTM(input_size=len(features), hidden_size=64, num_layers=2, output_size=1).to(device)
print(model)


## 4. Training Loop


In [ ]:
criterion = nn.BCEWithLogitsLoss() # Good for binary classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

print("Starting Training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Calculate training accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, Train Accuracy: {train_acc:.2f}%")


## 5. Evaluation


In [ ]:
model.eval()
correct = 0
total = 0

all_predictions = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = 100 * correct / total
print(f"Test Accuracy: {test_acc:.2f}%")

# Class distribution check
pred_ups = sum(all_predictions)
actual_ups = sum(all_labels)
print(f"Model predicted UP {pred_ups} times out of {len(all_predictions)} ({pred_ups/len(all_predictions)*100:.1f}%)")
print(f"Actual market went UP {actual_ups} times out of {len(all_labels)} ({actual_ups/len(all_labels)*100:.1f}%)")
